# 03. Modeling
Baseline, градиентный бустинг, тюнинг, MLflow.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


## 1. Загрузка processed данных и split


In [ ]:
import pandas as pd
import mlflow

from src.features import get_train_test_split
from src.train import (
    train_baseline, train_catboost, train_xgboost, train_lightgbm,
    tune_with_optuna, save_best_model, MLFLOW_EXPERIMENT,
)

df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'telco_processed.csv')
X_train, X_test, y_train, y_test = get_train_test_split(df)
mlflow.set_experiment(MLFLOW_EXPERIMENT)


## 2. Baseline: Logistic Regression


In [ ]:
lr_model, lr_metrics = train_baseline(X_train, y_train, X_test, y_test)
lr_metrics


## 3. Градиентный бустинг: CatBoost / XGBoost / LightGBM


In [ ]:
cb_model, cb_metrics = train_catboost(X_train, y_train, X_test, y_test)
xgb_model, xgb_metrics = train_xgboost(X_train, y_train, X_test, y_test)
lgbm_model, lgbm_metrics = train_lightgbm(X_train, y_train, X_test, y_test)
cb_metrics, xgb_metrics, lgbm_metrics


## 4. Сравнение моделей
Таблица метрик + сравнение class_weight vs SMOTE (TODO).


In [ ]:
comparison = pd.DataFrame({
    'logistic_regression': lr_metrics,
    'catboost': cb_metrics,
    'xgboost': xgb_metrics,
    'lightgbm': lgbm_metrics,
}).T
comparison.sort_values('pr_auc', ascending=False)


## 5. Тюнинг гиперпараметров (Optuna)


In [ ]:
best_params, best_pr_auc = tune_with_optuna(X_train, y_train, X_test, y_test, n_trials=30)
best_params, best_pr_auc


## 6. Сравнение запусков в MLflow


In [ ]:
runs = mlflow.search_runs(experiment_names=[MLFLOW_EXPERIMENT])
runs.sort_values('metrics.pr_auc', ascending=False)[['tags.mlflow.runName', 'metrics.pr_auc', 'metrics.roc_auc', 'metrics.recall']].head(10)


## 7. Сохранение лучшей модели


In [ ]:
# TODO: выбрать лучшую модель по сравнению выше и вызвать save_best_model(...)


## 8. Результаты
TODO: перенести итоговую таблицу метрик в README.
